In [2]:
import sqlite3
import pandas as pd

In [3]:
conn=sqlite3.connect(r"C:\Users\uygrg\OneDrive\Masaüstü\sqlite-sakila.db")

In [8]:
# Film tablosu
film_df = pd.read_sql_query("SELECT * FROM film", conn)

# Oyuncular
actor_df = pd.read_sql_query("SELECT * FROM actor", conn)

# Film-oyuncu ilişkisi
film_actor_df = pd.read_sql_query("SELECT * FROM film_actor", conn)

# Envanter
inventory_df = pd.read_sql_query("SELECT * FROM inventory", conn)

# Kiralama
#rental_df = pd.read_sql_query("SELECT * FROM rental", conn)

# Ödeme
payment_df = pd.read_sql_query("SELECT * FROM payment", conn)

# Müşteriler
customer_df = pd.read_sql_query("SELECT * FROM customer", conn)

# Film-kategori ve kategori tablosu
film_category_df = pd.read_sql_query("SELECT * FROM film_category", conn)
category_df = pd.read_sql_query("SELECT * FROM category", conn)


staff_df = pd.read_sql_query("SELECT * FROM staff;", conn)

address_df = pd.read_sql_query("SELECT * FROM address;", conn)
city_df = pd.read_sql_query("SELECT * FROM city;", conn)
customer_df = pd.read_sql_query("SELECT * FROM customer;", conn)
#rental_df = pd.read_sql_query("SELECT * FROM rental;", conn)
inventory_df = pd.read_sql_query("SELECT * FROM inventory;", conn)
film_df = pd.read_sql_query("SELECT * FROM film;", conn)
film_category_df = pd.read_sql_query("SELECT * FROM film_category;", conn)
category_df = pd.read_sql_query("SELECT * FROM category;", conn)
payment_df = pd.read_sql_query("SELECT * FROM payment;", conn)

store_df = pd.read_sql('SELECT * FROM store', conn)
staff_df = pd.read_sql('SELECT * FROM staff', conn)
rental_df = pd.read_sql('SELECT * FROM rental', conn) 
#rental_df = pd.read_sql_query("SELECT rental_id, inventory_id, rental_date, return_date, customer_id, staff_id FROM rental;", conn)


Soru 1: Tüm verileri göster

In [9]:
print(rental_df)

       rental_id              rental_date  inventory_id  customer_id  \
0              1  2005-05-24 22:53:30.000           367          130   
1              2  2005-05-24 22:54:33.000          1525          459   
2              3  2005-05-24 23:03:39.000          1711          408   
3              4  2005-05-24 23:04:41.000          2452          333   
4              5  2005-05-24 23:05:21.000          2079          222   
...          ...                      ...           ...          ...   
16039      16045  2005-08-23 22:25:26.000           772           14   
16040      16046  2005-08-23 22:26:47.000          4364           74   
16041      16047  2005-08-23 22:42:48.000          2088          114   
16042      16048  2005-08-23 22:43:07.000          2019          103   
16043      16049  2005-08-23 22:50:12.000          2666          393   

                   return_date  staff_id          last_update  
0      2005-05-26 22:04:30.000         1  2021-03-06 15:53:41  
1      

Soru 2: Her filmdeki oyuncuları listele

In [10]:
film_actor_df = film_df.merge(film_actor_df, on="film_id")
film_actor_df = film_actor_df.merge(actor_df, on="actor_id")
print(film_actor_df[["title", "first_name", "last_name"]])

                  title first_name  last_name
0      ACADEMY DINOSAUR   PENELOPE    GUINESS
1      ACADEMY DINOSAUR  CHRISTIAN      GABLE
2      ACADEMY DINOSAUR    LUCILLE      TRACY
3      ACADEMY DINOSAUR     SANDRA       PECK
4      ACADEMY DINOSAUR     JOHNNY       CAGE
...                 ...        ...        ...
5457  ZOOLANDER FICTION     WHOOPI       HURT
5458  ZOOLANDER FICTION       JADA      RYDER
5459          ZORRO ARK        IAN      TANDY
5460          ZORRO ARK       NICK  DEGENERES
5461          ZORRO ARK       LISA     MONROE

[5462 rows x 3 columns]


Soru 3: Her filmde kaç oyuncu oynadı?

In [11]:
film_actor_count = film_actor_df.groupby("title")["actor_id"].count().reset_index(name="actor_count")
print(film_actor_count)

                 title  actor_count
0     ACADEMY DINOSAUR           10
1       ACE GOLDFINGER            4
2     ADAPTATION HOLES            5
3     AFFAIR PREJUDICE            5
4          AFRICAN EGG            5
..                 ...          ...
992     YOUNG LANGUAGE            5
993         YOUTH KICK            5
994       ZHIVAGO CORE            6
995  ZOOLANDER FICTION            5
996          ZORRO ARK            3

[997 rows x 2 columns]


Soru 4: Her oyuncu kaç filmde oynadı? 

In [12]:
actor_movie_count = film_actor_df.groupby(["first_name", "last_name"])["film_id"].count().reset_index(name="movie_count")
print(actor_movie_count)

    first_name  last_name  movie_count
0         ADAM      GRANT           18
1         ADAM     HOPPER           22
2           AL    GARLAND           26
3         ALAN   DREYFUSS           27
4       ALBERT  JOHANSSON           33
..         ...        ...          ...
194       WILL     WILSON           31
195    WILLIAM    HACKMAN           27
196      WOODY    HOFFMAN           31
197      WOODY      JOLIE           31
198       ZERO       CAGE           25

[199 rows x 3 columns]


Soru 5: Envanterde olmayan filmler var mı ve varsa kaç tane?

In [13]:
not_in_inventory = film_df[~film_df["film_id"].isin(inventory_df["film_id"])]
print("Envanterde olmayan film sayısı:", len(not_in_inventory))

Envanterde olmayan film sayısı: 42


Soru 6: Kiralanabilir olan her filmin kaç kez kiralandığını ve toplam 
gelirlerini getirin

In [14]:
film_df = film_df[["film_id","title"]]
inventory_df = inventory_df[["inventory_id","film_id"]]
rental_dfReduced = rental_df[["rental_id","inventory_id"]]
payment_df = payment_df[["rental_id","amount"]]

film_rental = film_df.merge(inventory_df, on="film_id")
film_rental = film_rental.merge(rental_dfReduced, on="inventory_id")
film_rental = film_rental.merge(payment_df, on="rental_id")

film_rental_summary = film_rental.groupby("title").agg(
    rental_count=("rental_id", "count"),
    total_revenue=("amount", "sum")
).reset_index()

print(film_rental_summary)

                 title  rental_count  total_revenue
0     ACADEMY DINOSAUR            23          36.77
1       ACE GOLDFINGER             7          52.93
2     ADAPTATION HOLES            12          37.88
3     AFFAIR PREJUDICE            23          91.77
4          AFRICAN EGG            12          51.88
..                 ...           ...            ...
953     YOUNG LANGUAGE             7           6.93
954         YOUTH KICK             6          16.94
955       ZHIVAGO CORE             9          14.91
956  ZOOLANDER FICTION            17          73.83
957          ZORRO ARK            31         214.69

[958 rows x 3 columns]


Soru 7: Envanterde olmayan filmlerin kira oranlarını getirin

In [15]:
not_in_inventory_rates = not_in_inventory[["title", "rental_rate"]]
print(not_in_inventory_rates)
print(rental_df.columns)

                      title  rental_rate
13           ALICE FANTASIA         0.99
32              APOLLO TEEN         2.99
35           ARGONAUTS TOWN         0.99
37            ARK RIDGEMONT         0.99
40     ARSENIC INDEPENDENCE         0.99
86        BOONDOCK BALLROOM         0.99
107           BUTCH PANTHER         0.99
127           CATCH AMISTAD         0.99
143     CHINATOWN GLADIATOR         4.99
147          CHOCOLATE DUCK         2.99
170    COMMANDMENTS EXPRESS         4.99
191        CROSSING DIVORCE         4.99
194         CROWDS TELEMARK         4.99
197        CRYSTAL BREAKING         2.99
216              DAZED PUNK         4.99
220  DELIVERANCE MULHOLLAND         0.99
317       FIREHOUSE VIETNAM         0.99
324           FLOATS GARDEN         2.99
331   FRANKENSTEIN STRANGER         0.99
358      GLADIATOR WESTWARD         4.99
385               GUMP DATE         4.99
403           HATE HANDICAP         0.99
418             HOCUS FRIDA         2.99
494        KENTU

Soru 8: Birden fazla DVD'yi iade etmeyen kaç müşteri var?

In [16]:
not_returned = rental_df[rental_df["return_date"].isna()]
not_returned_count = not_returned.groupby("customer_id")["rental_id"].count().reset_index()
multiple_not_returned = not_returned_count[not_returned_count["rental_id"] > 1]
print("Birden fazla DVD’yi iade etmeyen müşteri sayısı:", len(multiple_not_returned))

Birden fazla DVD’yi iade etmeyen müşteri sayısı: 23


Soru 9: Her müşteri kaç film kiraladı?

In [47]:
customer_rental_count = customer_df.merge(rental_df, on="customer_id")
customer_rental_count = customer_rental_count.groupby(["first_name", "last_name"])["rental_id"].count().reset_index(name="rental_count")
print(customer_rental_count)

    first_name last_name  rental_count
0        AARON     SELBY            24
1         ADAM     GOOCH            22
2       ADRIAN     CLARY            19
3        AGNES    BISHOP            23
4         ALAN      KAHN            26
..         ...       ...           ...
594     WILLIE   MARKHAM            25
595      WILMA  RICHARDS            20
596    YOLANDA    WEAVER            27
597     YVONNE   WATKINS            21
598    ZACHARY      HITE            31

[599 rows x 3 columns]


Soru 10: Türlerine göre en çok kiralanan filmler ve bunlara ne 
kadar ödendi?

In [18]:
film_df = film_df[["film_id","title"]]
film_category_df = film_category_df[["film_id","category_id"]]
category_df = category_df[["category_id","name"]]
inventory_df = inventory_df[["inventory_id","film_id"]]
rental_dfReduced = rental_df[["rental_id","inventory_id"]]
payment_df = payment_df[["rental_id","amount"]]

# Merge işlemleri
film_cat = film_df.merge(film_category_df, on="film_id").merge(category_df, on="category_id")
film_rental_cat = film_cat.merge(inventory_df, on="film_id") \
                          .merge(rental_df, on="inventory_id") \
                          .merge(payment_df, on="rental_id")

film_rental_cat_summary = film_rental_cat.groupby(["title", "name"]).agg(
    rental_count=("rental_id", "count"),
    total_revenue=("amount", "sum")
).reset_index()

print(film_rental_cat_summary)

                 title         name  rental_count  total_revenue
0     ACADEMY DINOSAUR  Documentary            23          36.77
1       ACE GOLDFINGER       Horror             7          52.93
2     ADAPTATION HOLES  Documentary            12          37.88
3     AFFAIR PREJUDICE       Horror            23          91.77
4          AFRICAN EGG       Family            12          51.88
..                 ...          ...           ...            ...
953     YOUNG LANGUAGE  Documentary             7           6.93
954         YOUTH KICK        Music             6          16.94
955       ZHIVAGO CORE       Horror             9          14.91
956  ZOOLANDER FICTION     Children            17          73.83
957          ZORRO ARK       Comedy            31         214.69

[958 rows x 4 columns]


Soru 11: Tür ve Tarihe Göre Kiralama Sayısı ve Gelir 

In [19]:
rental_payment = rental_df[['rental_id', 'inventory_id', 'rental_date']].merge(
    payment_df[['rental_id', 'amount']], on='rental_id'
)

film_inventory_rental = film_df[['film_id', 'title']].merge(
    inventory_df[['inventory_id', 'film_id']], on='film_id'
).merge(rental_payment, on='inventory_id')

film_cat_rental = film_inventory_rental.merge(film_category_df[['film_id','category_id']], on='film_id') \
                                      .merge(category_df[['category_id','name']], on='category_id')

film_cat_rental['rental_date_only'] = pd.to_datetime(film_cat_rental['rental_date']).dt.date

genre_date_summary = film_cat_rental.groupby(['name', 'rental_date_only']).agg(
    rental_count=('rental_id','count'),
    total_revenue=('amount','sum')
).reset_index()

print(genre_date_summary)

       name rental_date_only  rental_count  total_revenue
0    Action       2005-05-25            10          46.90
1    Action       2005-05-26            15          80.85
2    Action       2005-05-27             9          37.91
3    Action       2005-05-28            20          68.80
4    Action       2005-05-29            12          42.88
..      ...              ...           ...            ...
628  Travel       2005-08-20            29         127.71
629  Travel       2005-08-21            39         163.61
630  Travel       2005-08-22            21         103.79
631  Travel       2005-08-23            27         126.73
632  Travel       2006-02-14            10          30.91

[633 rows x 4 columns]


Soru 12: Kiralanabilir Filmler İçin Türlerine Göre Her Filmin Kaç 
Kez Kiralandığı 

In [20]:
film_inventory_rental = film_df.merge(inventory_df, on="film_id").merge(rental_df, on="inventory_id")
film_cat_rental = film_inventory_rental.merge(film_category_df, on="film_id").merge(category_df, on="category_id")

genre_film_summary = film_cat_rental.groupby(["name","title"]).agg(
    rental_count=("rental_id","count")
).reset_index()

print(genre_film_summary)

       name                title  rental_count
0    Action         AMADEUS HOLY            21
1    Action      AMERICAN CIRCUS            22
2    Action   ANTITRUST TOMATOES            10
3    Action  BAREFOOT MANCHURIAN            18
4    Action         BERETS AGENT            21
..      ...                  ...           ...
953  Travel  VALENTINE VANISHING            12
954  Travel          WINDOW SIDE            12
955  Travel        WOLVES DESIRE            21
956  Travel        WORKER TARZAN            15
957  Travel  WORKING MICROCOSMOS            25

[958 rows x 3 columns]


Soru 13: En çok rafta bekleyen filmler

In [21]:
from datetime import datetime

film_inventory_rental = film_df.merge(inventory_df, on="film_id").merge(rental_df, on="inventory_id")
unreturned = film_inventory_rental[film_inventory_rental["return_date"].isnull()]
unreturned["days_on_shelf"] = (pd.to_datetime("today") - pd.to_datetime(unreturned["rental_date"])).dt.days

unreturned_sorted = unreturned[["title","days_on_shelf"]].sort_values(by="days_on_shelf", ascending=False)
print(unreturned_sorted)

                    title  days_on_shelf
16       ACADEMY DINOSAUR           7401
24         ACE GOLDFINGER           7223
61       AFFAIR PREJUDICE           7223
76            AFRICAN EGG           7223
224           ALI FOREVER           7223
...                   ...            ...
15615         WILD APOLLO           7223
15658         WINDOW SIDE           7223
15745        WOMEN DORADO           7223
15887  WORLD LEATHERNECKS           7223
15995        ZHIVAGO CORE           7223

[183 rows x 2 columns]


C:\Users\uygrg\AppData\Local\Temp\ipykernel_14124\3632848147.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  unreturned["days_on_shelf"] = (pd.to_datetime("today") - pd.to_datetime(unreturned["rental_date"])).dt.days


Soru 14: Geç, Erken ve Zamanında İade Edilen Kiralanmış Filmler

In [22]:
rental_duration_days = 3

rental_status = rental_df.copy()


rental_status['rental_date'] = pd.to_datetime(rental_status['rental_date'])
rental_status['return_date'] = pd.to_datetime(rental_status['return_date'])

rental_status['due_date'] = rental_status['rental_date'] + pd.to_timedelta(rental_duration_days, unit='d')

summary_returns = {
    "late_returns": ((rental_status['return_date'] > rental_status['due_date'])).sum(),
    "early_returns": ((rental_status['return_date'] < rental_status['due_date'])).sum(),
    "on_time_returns": ((rental_status['return_date'] == rental_status['due_date'])).sum()
}
print(summary_returns)

{'late_returns': np.int64(11472), 'early_returns': np.int64(4388), 'on_time_returns': np.int64(1)}


Soru 15: Hangi müşteri en çok DVD kiralamış?

In [23]:
customer_rental = customer_df.merge(rental_df, on="customer_id")
customer_summary = customer_rental.groupby(["first_name","last_name"]).agg(
    rental_count=("rental_id","count")
).reset_index()
top_customer = customer_summary.sort_values(by="rental_count", ascending=False).head(1)
print(top_customer)

    first_name last_name  rental_count
175    ELEANOR      HUNT            46


Soru 16: En popüler film kategorisi nedir? 

In [24]:
film_inv_rent_cat = film_df.merge(inventory_df, on="film_id") \
                           .merge(rental_df, on="inventory_id") \
                           .merge(film_category_df, on="film_id") \
                           .merge(category_df, on="category_id")
category_summary = film_inv_rent_cat.groupby("name").agg(
    rental_count=("rental_id","count")
).reset_index()
top_category = category_summary.sort_values("rental_count", ascending=False).head(1)
print(top_category)

      name  rental_count
14  Sports          1179


Soru 17: Hangi çalışan en çok kiralama işlemi gerçekleştirmiş?

In [25]:
staff_rental = staff_df.merge(rental_df, on="staff_id")
staff_summary = staff_rental.groupby(["first_name","last_name"]).agg(
    rental_count=("rental_id","count")
).reset_index()
top_staff = staff_summary.sort_values("rental_count", ascending=False).head(1)
print(top_staff)

  first_name last_name  rental_count
1       Mike   Hillyer          8040


Soru 18: En çok geliri hangi film getirmiş?

In [26]:
film_rental_payment = film_df.merge(inventory_df, on="film_id") \
                             .merge(rental_df, on="inventory_id") \
                             .merge(payment_df, on="rental_id")
film_revenue_summary = film_rental_payment.groupby("title").agg(
    total_revenue=("amount","sum")
).reset_index()
top_revenue_film = film_revenue_summary.sort_values("total_revenue", ascending=False).head(1)
print(top_revenue_film)

                title  total_revenue
841  TELEGRAPH VOYAGE         231.73


Soru 19: Her müşteri için toplam harcama miktarını bulun

In [27]:
cust_rent_pay = customer_df.merge(rental_df, on="customer_id").merge(payment_df, on="rental_id")
cust_spent_summary = cust_rent_pay.groupby(["first_name","last_name"]).agg(
    total_spent=("amount","sum")
).reset_index()
print(cust_spent_summary)

    first_name last_name  total_spent
0        AARON     SELBY       110.76
1         ADAM     GOOCH       101.78
2       ADRIAN     CLARY        74.81
3        AGNES    BISHOP        98.77
4         ALAN      KAHN       124.74
..         ...       ...          ...
594     WILLIE   MARKHAM       101.75
595      WILMA  RICHARDS        91.80
596    YOLANDA    WEAVER       110.73
597     YVONNE   WATKINS        92.79
598    ZACHARY      HITE       146.69

[599 rows x 3 columns]


Soru 20: Her kategorideki toplam kiralama sayısını ve gelirleri 
bulun 

In [28]:
film_inv_rent_cat_pay = film_df.merge(inventory_df, on="film_id") \
                               .merge(rental_df, on="inventory_id") \
                               .merge(payment_df, on="rental_id") \
                               .merge(film_category_df, on="film_id") \
                               .merge(category_df, on="category_id")
category_summary = film_inv_rent_cat_pay.groupby("name").agg(
    rental_count=("rental_id","count"),
    total_revenue=("amount","sum")
).reset_index()
print(category_summary)

           name  rental_count  total_revenue
0        Action          1112        4375.85
1     Animation          1166        4656.30
2      Children           945        3655.55
3      Classics           939        3639.59
4        Comedy           941        4383.58
5   Documentary          1050        4217.52
6         Drama          1060        4587.39
7        Family          1096        4226.07
8       Foreign          1033        4270.67
9         Games           969        4281.33
10       Horror           846        3722.54
11        Music           830        3417.72
12          New           940        4351.62
13       Sci-Fi          1101        4756.98
14       Sports          1179        5314.21
15       Travel           837        3549.64


Soru 21: En uzun süre kirada kalmış filmleri bulun

In [29]:
film_rental = film_df.merge(inventory_df, on="film_id").merge(rental_df, on="inventory_id")

film_rental_valid = film_rental[film_rental["return_date"].notna()]

film_rental_valid["rental_duration"] = (pd.to_datetime(film_rental_valid["return_date"]) - pd.to_datetime(film_rental_valid["rental_date"])).dt.days

max_rental = film_rental_valid.groupby("title")["rental_duration"].max().reset_index()
max_rental_sorted = max_rental.sort_values("rental_duration", ascending=False).head(1)
print(max_rental_sorted)

              title  rental_duration
14  ALLEY EVOLUTION                9


C:\Users\uygrg\AppData\Local\Temp\ipykernel_14124\921209632.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  film_rental_valid["rental_duration"] = (pd.to_datetime(film_rental_valid["return_date"]) - pd.to_datetime(film_rental_valid["rental_date"])).dt.days


Soru 22: En az kiralanan 5 film hangileridir?

In [30]:
film_rental_count = film_inventory_rental.groupby("title").agg(
    rental_count=("rental_id","count")
).reset_index()
least_rented = film_rental_count.sort_values("rental_count").head(5)
print(least_rented)

                 title  rental_count
866        TRAIN BUNCH             4
378     HARDLY ROBBERS             4
558        MIXED DOORS             4
315  FREEDOM CLEOPATRA             5
341        GLORY TRACY             5


Soru 24: En fazla kazanç sağlayan 5 müşteriyi bulun

In [31]:
customer_rental = customer_df.merge(rental_df, on="customer_id").merge(payment_df, on="rental_id")
top_customers = customer_rental.groupby(["first_name","last_name"]).agg(
    total_spent=("amount","sum")
).reset_index().sort_values("total_spent", ascending=False).head(5)
print(top_customers)

    first_name last_name  total_spent
318       KARL      SEAL       221.55
175    ELEANOR      HUNT       216.54
105      CLARA      SHAW       195.58
389     MARION    SNYDER       194.61
474     RHONDA   KENNEDY       194.61


Soru 25: Her filmin ortalama kiralanma süresini bulun

In [32]:
film_rental_avg = film_rental_valid.groupby("title")["rental_duration"].mean().reset_index()
film_rental_avg.rename(columns={"rental_duration":"avg_rental_duration"}, inplace=True)
print(film_rental_avg)

                 title  avg_rental_duration
0     ACADEMY DINOSAUR             4.545455
1       ACE GOLDFINGER             5.333333
2     ADAPTATION HOLES             2.833333
3     AFFAIR PREJUDICE             4.363636
4          AFRICAN EGG             6.727273
..                 ...                  ...
953     YOUNG LANGUAGE             4.000000
954         YOUTH KICK             4.666667
955       ZHIVAGO CORE             5.250000
956  ZOOLANDER FICTION             5.176471
957          ZORRO ARK             4.096774

[958 rows x 2 columns]


Soru 26: Her türde en popüler filmi bulun

In [33]:
film_cat_rental = film_df[['film_id','title']].merge(
    film_category_df[['film_id','category_id']], on='film_id'
).merge(
    category_df[['category_id','name']], on='category_id'
).merge(
    inventory_df[['inventory_id','film_id']], on='film_id'
).merge(
    rental_df[['rental_id','inventory_id']], on='inventory_id'
)

genre_film_counts = film_cat_rental.groupby(['name','title']).agg(
    rental_count=('rental_id','count')
).reset_index()

top_genre_films = genre_film_counts.sort_values(['name','rental_count'], ascending=[True,False]).groupby('name').head(1)
print("Soru 26: Her türde en popüler film")
print(top_genre_films)

Soru 26: Her türde en popüler film
            name                title  rental_count
46        Action  RUGRATS SHAKESPEARE            30
93     Animation       JUGGLER HARDLY            32
165     Children         ROBBERS JOON            31
231     Classics       TIMBERLAND SKY            31
292       Comedy            ZORRO ARK            31
353  Documentary            WIFE TURN            31
380        Drama         HOBBIT ALIEN            31
418       Family        APACHE DIVINE            31
536      Foreign     ROCKETEER MOTHER            33
568        Games       FORWARD TEMPLE            32
640       Horror         PULP BEVERLY            30
701        Music        SCALAWAG DUCK            32
757          New  RIDGEMONT SUBMARINE            32
802       Sci-Fi    GOODFELLAS SALUTE            31
852       Sports  GLEAMING JAWBREAKER            29
909       Travel   BUCKET BROTHERHOOD            34


Soru 27: Her türde en fazla gelir sağlayan filmi bulun

In [34]:
film_cat_rental_payment = film_cat_rental.merge(
    payment_df[['rental_id','amount']], on='rental_id'
)

genre_film_revenue = film_cat_rental_payment.groupby(['name','title']).agg(
    total_revenue=('amount','sum')
).reset_index()

top_genre_revenue = genre_film_revenue.sort_values(['name','total_revenue'], ascending=[True,False]).groupby('name').head(1)
print("\nSoru 27: Her türde en fazla gelir sağlayan film")
print(top_genre_revenue)


Soru 27: Her türde en fazla gelir sağlayan film
            name                title  total_revenue
24        Action     FOOL MOCKINGBIRD         175.77
74     Animation         DOGMA FAMILY         178.70
125     Children  BACKLASH UNDEFEATED         158.81
229     Classics          STEEL SANTA         141.77
292       Comedy            ZORRO ARK         214.69
353  Documentary            WIFE TURN         223.69
408        Drama         TORQUE BOUND         198.72
467       Family     RANGE MOONWALKER         179.73
514      Foreign       INNOCENT USUAL         191.74
584        Games       MASSACRE USUAL         179.70
633       Horror           LOLA AGENT         159.76
705        Music     TELEGRAPH VOYAGE         231.73
745          New          MAIDEN HOME         163.76
802       Sci-Fi    GOODFELLAS SALUTE         209.69
887       Sports       SATURDAY LAMBS         204.72
909       Travel   BUCKET BROTHERHOOD         180.66


Soru 28: En çok DVD iade etmeyen müşteriyi bulun

In [35]:
unreturned = rental_df[rental_df["return_date"].isna()].merge(customer_df, on="customer_id")
top_unreturned = unreturned.groupby(["first_name","last_name"]).agg(
    unreturned_count=("rental_id","count")
).reset_index().sort_values("unreturned_count", ascending=False).head(1)
print(top_unreturned)

    first_name last_name  unreturned_count
144      TAMMY   SANDERS                 3


Soru 29: En fazla kiralama yapan 5 çalışanı bulun

In [36]:
staff_rental = rental_df.merge(staff_df, on="staff_id")
top_staff = staff_rental.groupby(["first_name","last_name"]).agg(
    rental_count=("rental_id","count")
).reset_index().sort_values("rental_count", ascending=False).head(5)
print(top_staff)

  first_name last_name  rental_count
1       Mike   Hillyer          8040
0        Jon  Stephens          8004


Soru 30: En fazla kiralama yapan 5 müşteri hangi şubeden 
kiralama yapmış?

In [44]:
print(customer_rental)
print(rental_df)
print(inventory_df)
customer_rental.rename(columns={"store_id_y":"store_id"}, inplace=True)
customer_renta_merged = customer_rental.merge(inventory_df, on="inventory_id")

top_cust_store = customer_renta_merged.groupby(
    ["first_name","last_name","store_id"], as_index=False
).agg(rental_count=("store_id","count")).sort_values("rental_count", ascending=False).head(5)

print(top_cust_store)

       customer_id  store_id first_name last_name  \
0                1         1       MARY     SMITH   
1                1         1       MARY     SMITH   
2                1         1       MARY     SMITH   
3                1         1       MARY     SMITH   
4                1         1       MARY     SMITH   
...            ...       ...        ...       ...   
16039          599         2     AUSTIN   CINTRON   
16040          599         2     AUSTIN   CINTRON   
16041          599         2     AUSTIN   CINTRON   
16042          599         2     AUSTIN   CINTRON   
16043          599         2     AUSTIN   CINTRON   

                                   email  address_id active  \
0          MARY.SMITH@sakilacustomer.org           5      1   
1          MARY.SMITH@sakilacustomer.org           5      1   
2          MARY.SMITH@sakilacustomer.org           5      1   
3          MARY.SMITH@sakilacustomer.org           5      1   
4          MARY.SMITH@sakilacustomer.org        

Soru 31: Her türde en az kiralanan filmi bulun

In [ ]:
film_cat_rental = film_df[['film_id','title']].merge(
    film_category_df[['film_id','category_id']], on='film_id'
).merge(
    category_df[['category_id','name']], on='category_id'
).merge(
    inventory_df[['inventory_id','film_id']], on='film_id'
).merge(
    rental_df[['rental_id','inventory_id']], on='inventory_id'
)

genre_film_counts = film_cat_rental.groupby(['name','title']).agg(
    rental_count=('rental_id','count')
).reset_index()

# Her türde en az kiralanan film
least_rented = genre_film_counts.sort_values(['name','rental_count'], ascending=[True,True]).groupby('name').head(1)
print("Soru 31: Her türde en az kiralanan film")
print(least_rented)

Soru 32: En çok kiralama yapan 5 müşteri hangi şehirde? 

In [136]:
cust_city_rental = rental_df.merge(customer_df, on='customer_id').merge(
    address_df[['address_id','city_id']], on='address_id'
).merge(
    city_df[['city_id','city']], on='city_id'
)

top_cust_city = cust_city_rental.groupby(['city','first_name','last_name']).agg(
    rental_count=('rental_id','count')
).reset_index().sort_values('rental_count', ascending=False).head(5)

print("Soru 32: En çok kiralama yapan 5 müşteri hangi şehirde")
print(top_cust_city)

Soru 32: En çok kiralama yapan 5 müşteri hangi şehirde
            city first_name last_name  rental_count
433  Saint-Denis    ELEANOR      HUNT            46
96    Cape Coral       KARL      SEAL            45
333    Molodetno      CLARA      SHAW            42
518        Tanza     MARCIA      DEAN            42
103     Changhwa      TAMMY   SANDERS            41


Soru 33: En çok kazanç sağlayan 5 müşteriyi hangi şehirde bulun?

In [137]:
cust_city_payment = rental_df.merge(customer_df, on='customer_id').merge(
    address_df[['address_id','city_id']], on='address_id'
).merge(
    city_df[['city_id','city']], on='city_id'
).merge(
    payment_df[['rental_id','amount']], on='rental_id'
)

top_cust_city_revenue = cust_city_payment.groupby(['city','first_name','last_name']).agg(
    total_spent=('amount','sum')
).reset_index().sort_values('total_spent', ascending=False).head(5)

print("Soru 33: En çok kazanç sağlayan 5 müşteri hangi şehirde")
print(top_cust_city_revenue)

Soru 33: En çok kazanç sağlayan 5 müşteri hangi şehirde
                    city first_name last_name  total_spent
96            Cape Coral       KARL      SEAL       221.55
433          Saint-Denis    ELEANOR      HUNT       216.54
333            Molodetno      CLARA      SHAW       195.58
23             Apeldoorn     RHONDA   KENNEDY       194.61
447  Santa Brbara dOeste     MARION    SNYDER       194.61


Soru 34: En çok kiralanan 5 filmi hangi şehirde bulun?

In [138]:
city_film_rental = rental_df.merge(customer_df, on='customer_id').merge(
    address_df[['address_id','city_id']], on='address_id'
).merge(
    city_df[['city_id','city']], on='city_id'
).merge(
    inventory_df[['inventory_id','film_id']], on='inventory_id'
).merge(
    film_df[['film_id','title']], on='film_id'
)

top_city_film = city_film_rental.groupby(['city','title']).agg(
    rental_count=('rental_id','count')
).reset_index().sort_values('rental_count', ascending=False).head(5)

print("Soru 34: En çok kiralanan 5 filmi hangi şehirde")
print(top_city_film)

Soru 34: En çok kiralanan 5 filmi hangi şehirde
                        city              title  rental_count
7828                    Lima    DISCIPLE MOTHER             3
7308                  Kurgan   DETECTIVE VISION             3
12735               Sorocaba    CADDYSHACK JEDI             3
14182                Trshavn  FLATLINERS KILLER             3
3793   Donostia-San Sebastin    REIGN GENTLEMEN             2


Soru 35: En az kiralanan 5 filmi hangi şehirde bulun?

In [139]:
least_city_film = city_film_rental.groupby(['city','title']).agg(
    rental_count=('rental_id','count')
).reset_index().sort_values('rental_count', ascending=True).head(5)

print("Soru 35: En az kiralanan 5 filmi hangi şehirde")
print(least_city_film)

Soru 35: En az kiralanan 5 filmi hangi şehirde
                     city                title  rental_count
15826     s-Hertogenbosch         WORST BANGER             1
17     A Corua (La Corua)      JEEPERS WEDDING             1
18     A Corua (La Corua)  LADYBUGS ARMAGEDDON             1
19     A Corua (La Corua)     LOVERBOY ATTACKS             1
20     A Corua (La Corua)       MASSACRE USUAL             1


Soru 36: En çok kazanç sağlayan 5 filmi hangi şehirde bulun?

In [ ]:
city_film_payment = city_film_rental.merge(
    payment_df[['rental_id','amount']], on='rental_id'
)

top_city_film_revenue = city_film_payment.groupby(['city','title']).agg(
    total_revenue=('amount','sum')
).reset_index().sort_values('total_revenue', ascending=False).head(5)

print("Soru 36: En çok kazanç sağlayan 5 filmi hangi şehirde")
print(top_city_film_revenue)

Soru 37: En az kazanç sağlayan 5 filmi hangi şehirde bulun?

In [141]:
least_city_film_revenue = city_film_payment.groupby(['city','title']).agg(
    total_revenue=('amount','sum')
).reset_index().sort_values('total_revenue', ascending=True).head(5)

print("Soru 37: En az kazanç sağlayan 5 filmi hangi şehirde")
print(least_city_film_revenue)

Soru 37: En az kazanç sağlayan 5 filmi hangi şehirde
          city            title  total_revenue
4152  Erlangen  STATE WASTELAND            0.0
4317      Faaa  MOTIONS DETAILS            0.0
4360  Fengshan  VANISHED GARDEN            0.0
5924    Jaipur  CHAMBER ITALIAN            0.0
2786  Changhwa  SLEEPY JAPANESE            0.0


Soru 38: En fazla kiralama yapan müşteri hangi filmleri kiralamış?

In [142]:
cust_rental_count = rental_df.groupby('customer_id').agg(rental_count=('rental_id','count')).reset_index()
top_customer_id = cust_rental_count.sort_values('rental_count', ascending=False).head(1)['customer_id'].iloc[0]

top_customer_rentals = rental_df[rental_df['customer_id']==top_customer_id].merge(
    customer_df[['customer_id','first_name','last_name']], on='customer_id'
).merge(
    inventory_df[['inventory_id','film_id']], on='inventory_id'
).merge(
    film_df[['film_id','title']], on='film_id'
).merge(
    film_category_df[['film_id','category_id']], on='film_id'
).merge(
    category_df[['category_id','name']], on='category_id'
)

top_customer_rentals_summary = top_customer_rentals.groupby(['first_name','last_name','name']).agg(
    rental_count=('rental_id','count')
).reset_index()

print(top_customer_rentals_summary)

   first_name last_name         name  rental_count
0     ELEANOR      HUNT       Action             3
1     ELEANOR      HUNT     Classics             4
2     ELEANOR      HUNT       Comedy             2
3     ELEANOR      HUNT  Documentary             3
4     ELEANOR      HUNT        Drama             1
5     ELEANOR      HUNT       Family             6
6     ELEANOR      HUNT      Foreign             4
7     ELEANOR      HUNT        Games             3
8     ELEANOR      HUNT       Horror             4
9     ELEANOR      HUNT        Music             1
10    ELEANOR      HUNT          New             2
11    ELEANOR      HUNT       Sci-Fi             7
12    ELEANOR      HUNT       Sports             1
13    ELEANOR      HUNT       Travel             5


Soru 40: En çok kazanç sağlayan müşteri hangi filmleri kiralamış?

In [143]:
rental_payment = rental_df[['rental_id','customer_id']].merge(payment_df[['rental_id','amount']], on='rental_id')

cust_payment_sum = rental_payment.groupby('customer_id', as_index=False)['amount'].sum()
cust_payment_sum = cust_payment_sum.rename(columns={'amount':'total_spent'})

top_customer_id = cust_payment_sum.sort_values('total_spent', ascending=False).iloc[0]['customer_id']

top_customer_payments = rental_df[rental_df['customer_id']==top_customer_id] \
    .merge(customer_df[['customer_id','first_name','last_name']], on='customer_id') \
    .merge(inventory_df[['inventory_id','film_id']], on='inventory_id') \
    .merge(film_df[['film_id','title']], on='film_id') \
    .merge(payment_df[['rental_id','amount']], on='rental_id')

top_customer_summary = top_customer_payments.groupby(
    ['first_name','last_name','title'], as_index=False
)['amount'].sum().rename(columns={'amount':'total_spent'})

print(top_customer_summary)

   first_name last_name                  title  total_spent
0        KARL      SEAL          BETRAYED REAR         4.99
1        KARL      SEAL       BIKINI BORROWERS         6.99
2        KARL      SEAL         BLUES INSTINCT         2.99
3        KARL      SEAL    BRINGING HYSTERICAL         2.99
4        KARL      SEAL         CYCLONE FAMILY         4.99
5        KARL      SEAL             DATE SPEED         3.99
6        KARL      SEAL           DESIRE ALIEN         2.99
7        KARL      SEAL       DESTINY SATURDAY         4.99
8        KARL      SEAL             DUMBO LUST         0.99
9        KARL      SEAL           DURHAM PANKY         4.99
10       KARL      SEAL       ENGLISH BULWORTH         5.99
11       KARL      SEAL         FIDELITY DEVIL         4.99
12       KARL      SEAL       FOOL MOCKINGBIRD        10.99
13       KARL      SEAL  FORRESTER COMANCHEROS         4.99
14       KARL      SEAL            GANGS PRIDE         2.99
15       KARL      SEAL        GOLDMINE 

Soru 41: En az kazanç sağlayan müşteri hangi filmleri kiralamış?

In [144]:
rental_payment = rental_df.merge(payment_df[['rental_id','amount']], on='rental_id')

cust_payment_sum = rental_payment.groupby('customer_id', as_index=False)['amount'].sum().rename(columns={'amount':'total_spent'})

bottom_customer_id = cust_payment_sum.sort_values('total_spent', ascending=True).iloc[0]['customer_id']

bottom_customer_rentals = rental_df[rental_df['customer_id']==bottom_customer_id] \
    .merge(customer_df[['customer_id','first_name','last_name']], on='customer_id') \
    .merge(inventory_df[['inventory_id','film_id']], on='inventory_id') \
    .merge(film_df[['film_id','title']], on='film_id') \
    .merge(payment_df[['rental_id','amount']], on='rental_id')

bottom_customer_summary = bottom_customer_rentals.groupby(['first_name','last_name','title'], as_index=False)['amount'].sum().rename(columns={'amount':'total_spent'})

print(bottom_customer_summary)

   first_name last_name                   title  total_spent
0    CAROLINE    BOWMAN         ARMAGEDDON LOST         0.99
1    CAROLINE    BOWMAN        CAMELOT VACATION         0.99
2    CAROLINE    BOWMAN         DISCIPLE MOTHER         5.99
3    CAROLINE    BOWMAN        EMPIRE MALKOVICH         0.99
4    CAROLINE    BOWMAN          FRENCH HOLIDAY         7.99
5    CAROLINE    BOWMAN       GOODFELLAS SALUTE         4.99
6    CAROLINE    BOWMAN       HOOSIERS BIRDCAGE         4.99
7    CAROLINE    BOWMAN         ILLUSION AMELIE         0.99
8    CAROLINE    BOWMAN    IMPOSSIBLE PREJUDICE         5.99
9    CAROLINE    BOWMAN              JADE BUNCH         2.99
10   CAROLINE    BOWMAN         KILLER INNOCENT         2.99
11   CAROLINE    BOWMAN              MASK PEACH         3.99
12   CAROLINE    BOWMAN  RESURRECTION SILVERADO         0.99
13   CAROLINE    BOWMAN             TEXAS WATCH         0.99
14   CAROLINE    BOWMAN         WHISPERER GIANT         4.99


Soru 42: En az kazanç sağlayan müşteri hangi türde en fazla film 
kiralamış?

In [145]:
bottom_customer_id = cust_payment_sum.sort_values('total_spent', ascending=True).iloc[0]['customer_id']

bottom_customer_genre = rental_df[rental_df['customer_id']==bottom_customer_id] \
    .merge(inventory_df[['inventory_id','film_id']], on='inventory_id') \
    .merge(film_category_df, on='film_id') \
    .merge(category_df[['category_id','name']], on='category_id') \
    .merge(customer_df[['customer_id','first_name','last_name']], on='customer_id')

bottom_customer_genre_summary = bottom_customer_genre.groupby(['first_name','last_name','name'], as_index=False)['rental_id'].count().rename(columns={'rental_id':'rental_count'}).sort_values('rental_count', ascending=False).head(1)

print(bottom_customer_genre_summary)

  first_name last_name    name  rental_count
6   CAROLINE    BOWMAN  Sci-Fi             4


Soru 43: En çok kiralanan film hangi çalışan tarafından 
kiralanmış?

In [146]:
film_rental_counts = rental_df.merge(inventory_df[['inventory_id','film_id']], on='inventory_id') \
                              .groupby('film_id')['rental_id'].count().reset_index()

top_film_id = film_rental_counts.sort_values('rental_id', ascending=False).iloc[0]['film_id']

staff_film_rentals = rental_df[rental_df['inventory_id'].isin(inventory_df[inventory_df['film_id']==top_film_id]['inventory_id'])] \
    .merge(staff_df[['staff_id','first_name','last_name']], on='staff_id') \
    .merge(inventory_df[['inventory_id','film_id']], on='inventory_id') \
    .merge(film_df[['film_id','title']], on='film_id')

staff_film_summary = staff_film_rentals.groupby(['first_name','last_name','title'], as_index=False)['rental_id'].count().rename(columns={'rental_id':'rental_count'})
print(staff_film_summary)

  first_name last_name               title  rental_count
0        Jon  Stephens  BUCKET BROTHERHOOD            18
1       Mike   Hillyer  BUCKET BROTHERHOOD            16


Soru 44: En az kiralanan film hangi çalışan tarafından kiralanmış?

In [147]:
film_rental_counts = rental_df.merge(inventory_df[['inventory_id','film_id']], on='inventory_id') \
                              .groupby('film_id', as_index=False)['rental_id'].count()

bottom_film_id = film_rental_counts.sort_values('rental_id', ascending=True).iloc[0]['film_id']

staff_film_rentals = rental_df[rental_df['inventory_id'].isin(inventory_df[inventory_df['film_id']==bottom_film_id]['inventory_id'])] \
    .merge(staff_df[['staff_id','first_name','last_name']], on='staff_id') \
    .merge(inventory_df[['inventory_id','film_id']], on='inventory_id') \
    .merge(film_df[['film_id','title']], on='film_id')

staff_film_summary = staff_film_rentals.groupby(['first_name','last_name','title'], as_index=False)['rental_id'].count().rename(columns={'rental_id':'rental_count'})
print(staff_film_summary)

  first_name last_name        title  rental_count
0        Jon  Stephens  TRAIN BUNCH             4


Soru 45: En çok kazanç sağlayan film hangi çalışan tarafından 
kiralanmış?

In [148]:
film_revenue = rental_df.merge(inventory_df[['inventory_id','film_id']], on='inventory_id') \
                        .merge(payment_df[['rental_id','amount']], on='rental_id') \
                        .groupby('film_id', as_index=False)['amount'].sum()

top_film_id = film_revenue.sort_values('amount', ascending=False).iloc[0]['film_id']

staff_film_payments = rental_df[rental_df['inventory_id'].isin(inventory_df[inventory_df['film_id']==top_film_id]['inventory_id'])] \
    .merge(staff_df[['staff_id','first_name','last_name']], on='staff_id') \
    .merge(inventory_df[['inventory_id','film_id']], on='inventory_id') \
    .merge(film_df[['film_id','title']], on='film_id') \
    .merge(payment_df[['rental_id','amount']], on='rental_id')

staff_film_summary = staff_film_payments.groupby(['first_name','last_name','title'], as_index=False)['amount'].sum().rename(columns={'amount':'total_revenue'})
print(staff_film_summary)

  first_name last_name             title  total_revenue
0        Jon  Stephens  TELEGRAPH VOYAGE         147.83
1       Mike   Hillyer  TELEGRAPH VOYAGE          83.90


Soru 46: En az kazanç sağlayan film hangi çalışan tarafından 
kiralanmış?

In [149]:
film_revenue = rental_df.merge(inventory_df[['inventory_id','film_id']], on='inventory_id') \
                        .merge(payment_df[['rental_id','amount']], on='rental_id') \
                        .groupby('film_id', as_index=False)['amount'].sum()

bottom_film_id = film_revenue.sort_values('amount', ascending=True).iloc[0]['film_id']

staff_film_payments = rental_df[rental_df['inventory_id'].isin(inventory_df[inventory_df['film_id']==bottom_film_id]['inventory_id'])] \
    .merge(staff_df[['staff_id','first_name','last_name']], on='staff_id') \
    .merge(inventory_df[['inventory_id','film_id']], on='inventory_id') \
    .merge(film_df[['film_id','title']], on='film_id') \
    .merge(payment_df[['rental_id','amount']], on='rental_id')

staff_film_summary = staff_film_payments.groupby(['first_name','last_name','title'], as_index=False)['amount'].sum().rename(columns={'amount':'total_revenue'})
print(staff_film_summary)

  first_name last_name        title  total_revenue
0        Jon  Stephens  TEXAS WATCH           2.97
1       Mike   Hillyer  TEXAS WATCH           2.97


Soru 47: En çok kiralanan film hangi mağazada kiralanmış?

In [150]:
film_rental_counts = rental_df.merge(inventory_df[['inventory_id','film_id']], on='inventory_id') \
                              .groupby('film_id', as_index=False)['rental_id'].count()

top_film_id = film_rental_counts.sort_values('rental_id', ascending=False).iloc[0]['film_id']

store_film_rentals = rental_df[rental_df['inventory_id'].isin(inventory_df[inventory_df['film_id']==top_film_id]['inventory_id'])] \
    .merge(staff_df[['staff_id','store_id']], on='staff_id') \
    .merge(inventory_df[['inventory_id','film_id']], on='inventory_id') \
    .merge(film_df[['film_id','title']], on='film_id')

store_film_summary = store_film_rentals.groupby(['store_id','title'], as_index=False)['rental_id'].count().rename(columns={'rental_id':'rental_count'})
print(store_film_summary)

   store_id               title  rental_count
0         1  BUCKET BROTHERHOOD            16
1         2  BUCKET BROTHERHOOD            18


Soru 48: En az kiralanan film hangi mağazada kiralanmış?

In [151]:
bottom_film_id = film_rental_counts.sort_values('rental_id', ascending=True).iloc[0]['film_id']

store_film_rentals = rental_df[rental_df['inventory_id'].isin(inventory_df[inventory_df['film_id']==bottom_film_id]['inventory_id'])] \
    .merge(staff_df[['staff_id','store_id']], on='staff_id') \
    .merge(inventory_df[['inventory_id','film_id']], on='inventory_id') \
    .merge(film_df[['film_id','title']], on='film_id')

store_film_summary = store_film_rentals.groupby(['store_id','title'], as_index=False)['rental_id'].count().rename(columns={'rental_id':'rental_count'})
print(store_film_summary)

   store_id        title  rental_count
0         2  TRAIN BUNCH             4


Soru 49: En çok kazanç sağlayan film hangi mağazada kiralanmış? 

In [152]:
film_revenue = rental_df.merge(inventory_df[['inventory_id','film_id']], on='inventory_id') \
                        .merge(payment_df[['rental_id','amount']], on='rental_id') \
                        .groupby('film_id', as_index=False)['amount'].sum()

top_film_id = film_revenue.sort_values('amount', ascending=False).iloc[0]['film_id']

store_film_payments = rental_df[rental_df['inventory_id'].isin(inventory_df[inventory_df['film_id']==top_film_id]['inventory_id'])] \
    .merge(staff_df[['staff_id','store_id']], on='staff_id') \
    .merge(inventory_df[['inventory_id','film_id']], on='inventory_id') \
    .merge(film_df[['film_id','title']], on='film_id') \
    .merge(payment_df[['rental_id','amount']], on='rental_id')

store_film_summary = store_film_payments.groupby(['store_id','title'], as_index=False)['amount'].sum().rename(columns={'amount':'total_revenue'})
print(store_film_summary)

   store_id             title  total_revenue
0         1  TELEGRAPH VOYAGE          83.90
1         2  TELEGRAPH VOYAGE         147.83


Soru 50: En az kazanç sağlayan film hangi mağazada kiralanmış?

In [153]:
bottom_film_id = film_revenue.sort_values('amount', ascending=True).iloc[0]['film_id']

store_film_payments = rental_df[rental_df['inventory_id'].isin(inventory_df[inventory_df['film_id']==bottom_film_id]['inventory_id'])] \
    .merge(staff_df[['staff_id','store_id']], on='staff_id') \
    .merge(inventory_df[['inventory_id','film_id']], on='inventory_id') \
    .merge(film_df[['film_id','title']], on='film_id') \
    .merge(payment_df[['rental_id','amount']], on='rental_id')

store_film_summary = store_film_payments.groupby(['store_id','title'], as_index=False)['amount'].sum().rename(columns={'amount':'total_revenue'})
print(store_film_summary)

   store_id        title  total_revenue
0         1  TEXAS WATCH           2.97
1         2  TEXAS WATCH           2.97


Soru 51: Müşterilerin kiraladıkları filmlerin toplam kiralama süresi ne kadar?

In [154]:
rental_df['rental_days'] = (pd.to_datetime(rental_df['return_date']) - pd.to_datetime(rental_df['rental_date'])).dt.days
customer_rental_days = rental_df.merge(customer_df[['customer_id','first_name','last_name']], on='customer_id') \
    .groupby(['first_name','last_name'], as_index=False)['rental_days'].sum().rename(columns={'rental_days':'total_rental_days'})
print(customer_rental_days)

    first_name last_name  total_rental_days
0        AARON     SELBY              109.0
1         ADAM     GOOCH              106.0
2       ADRIAN     CLARY               90.0
3        AGNES    BISHOP               97.0
4         ALAN      KAHN              131.0
..         ...       ...                ...
594     WILLIE   MARKHAM              114.0
595      WILMA  RICHARDS              106.0
596    YOLANDA    WEAVER              123.0
597     YVONNE   WATKINS               83.0
598    ZACHARY      HITE              157.0

[599 rows x 3 columns]


Soru 52: En çok kiralanan türdeki filmler hangileri?

In [155]:
film_category_full = film_df[['film_id','title']].merge(film_category_df[['film_id','category_id']], on='film_id') \
    .merge(category_df[['category_id','name']], on='category_id')

genre_rental_max = rental_df.merge(inventory_df[['inventory_id','film_id']], on='inventory_id') \
    .merge(film_category_full, on='film_id') \
    .groupby(['name','title'], as_index=False).size().rename(columns={'size':'rental_count'}) \
    .sort_values('rental_count', ascending=False).head(1)

print("En çok kiralanan tür ve film:")
print(genre_rental_max)

En çok kiralanan tür ve film:
       name               title  rental_count
909  Travel  BUCKET BROTHERHOOD            34


Soru 53: En az kiralanan türdeki filmler hangileri?

In [156]:
genre_rental_full = rental_df.merge(inventory_df[['inventory_id','film_id']], on='inventory_id') \
    .merge(film_category_full, on='film_id') \
    .groupby(['name','title'], as_index=False).size().rename(columns={'size':'rental_count'})

genre_rental_min = genre_rental_full.sort_values('rental_count', ascending=True).head(1)

print("En az kiralanan tür ve film:")
print(genre_rental_min)

En az kiralanan tür ve film:
        name        title  rental_count
525  Foreign  MIXED DOORS             4


Soru 54: Her müşteri için toplam ödeme miktarını bulun.

In [157]:
customer_payment = (
    payment_df[['rental_id','amount']] 
    .merge(rental_df[['rental_id','customer_id']], on='rental_id')
    .merge(customer_df[['customer_id','first_name','last_name']], on='customer_id')
    .groupby(['first_name','last_name'], as_index=False)['amount'].sum()
    .rename(columns={'amount':'total_payment'})
)

print(customer_payment.head())

  first_name last_name  total_payment
0      AARON     SELBY         110.76
1       ADAM     GOOCH         101.78
2     ADRIAN     CLARY          74.81
3      AGNES    BISHOP          98.77
4       ALAN      KAHN         124.74


Soru 55: Hangi filmler en uzun süre kiralanmış?

In [158]:
film_rental_days = rental_df.merge(inventory_df[['inventory_id','film_id']], on='inventory_id') \
    .merge(film_df[['film_id','title']], on='film_id')
film_rental_days['rental_days'] = (pd.to_datetime(film_rental_days['return_date']) - pd.to_datetime(film_rental_days['rental_date'])).dt.days
longest_rental = film_rental_days.groupby('title')['rental_days'].max().reset_index().sort_values('rental_days', ascending=False).head()
print(longest_rental)

                title  rental_days
14    ALLEY EVOLUTION          9.0
957         ZORRO ARK          9.0
0    ACADEMY DINOSAUR          9.0
15         ALONE TRIP          9.0
2    ADAPTATION HOLES          9.0


Soru 56: Hangi filmler en kısa süre kiralanmış?

In [159]:
shortest_rental = film_rental_days.groupby('title')['rental_days'].min().reset_index().sort_values('rental_days', ascending=True).head()
print(shortest_rental)

                    title  rental_days
944          WORDS HUNTER          0.0
943             WONKA SEA          0.0
942  WONDERLAND CHRISTMAS          0.0
941        WONDERFUL DROP          0.0
940             WON DARES          0.0


Soru 57: Müşterilerin kiraladığı filmler için ortalama ödeme miktarını bulun. 

In [160]:
avg_payment = (
    payment_df[['rental_id','amount']]
    .merge(rental_df[['rental_id','customer_id']], on='rental_id')
    .merge(customer_df[['customer_id','first_name','last_name']], on='customer_id')
    .groupby(['first_name','last_name'], as_index=False)['amount'].mean()
    .rename(columns={'amount':'avg_payment'})
)

print(avg_payment.head())

  first_name last_name  avg_payment
0      AARON     SELBY     4.615000
1       ADAM     GOOCH     4.626364
2     ADRIAN     CLARY     3.937368
3      AGNES    BISHOP     4.294348
4       ALAN      KAHN     4.797692


Soru 58: En çok kiralanan filmlerin ortalama kiralama süresi nedir?

In [161]:
film_rental_avg = film_rental_days.groupby('title')['rental_days'].mean().reset_index()
top_film = film_rental_days.groupby('title')['rental_id'].count().reset_index().sort_values('rental_id', ascending=False).head(1)['title'].iloc[0]
print(film_rental_avg[film_rental_avg['title']==top_film])

                 title  rental_days
96  BUCKET BROTHERHOOD     4.411765


Soru 59: En az kiralanan filmlerin ortalama kiralama süresi nedir?

In [162]:
bottom_film = film_rental_days.groupby('title')['rental_id'].count().reset_index().sort_values('rental_id', ascending=True).head(1)['title'].iloc[0]
print(film_rental_avg[film_rental_avg['title']==bottom_film])

           title  rental_days
866  TRAIN BUNCH          3.0


Soru 60: En çok kazanç sağlayan müşterilerin ortalama ödeme miktarı nedir?

In [163]:
payment_clean = payment_df[['payment_id', 'rental_id', 'amount', 'customer_id']].copy()

pay_rental = payment_clean.merge(rental_df[['rental_id','customer_id']], on='rental_id', how='left', suffixes=('_pay','_rent'))

pay_rental['customer_id'] = pay_rental['customer_id_rent']

top_customers_avg_payment = pay_rental.merge(
    customer_df[['customer_id','first_name','last_name']],
    on='customer_id',
    how='left'
).groupby(['first_name','last_name'], as_index=False) \
 .agg(avg_payment=('amount','mean'), total_payment=('amount','sum')) \
 .sort_values('total_payment', ascending=False) \
 .head(10)

print(top_customers_avg_payment)

    first_name last_name  avg_payment  total_payment
318       KARL      SEAL     4.923333         221.55
175    ELEANOR      HUNT     4.707391         216.54
105      CLARA      SHAW     4.656667         195.58
389     MARION    SNYDER     4.990000         194.61
474     RHONDA   KENNEDY     4.990000         194.61
556      TOMMY   COLLAZO     4.911053         186.62
590     WESLEY      BULL     4.440000         177.60
551        TIM      CARY     4.502821         175.61
379     MARCIA      DEAN     4.180476         175.58
21         ANA   BRADLEY     5.137059         174.66


Soru 61: En az kazanç sağlayan müşterilerin ortalama ödeme miktarı nedir?

In [164]:
pay_rental = payment_df[['rental_id','amount']].merge(rental_df[['rental_id','customer_id']], on='rental_id')

pay_rental = pay_rental.merge(customer_df[['customer_id','first_name','last_name']], on='customer_id')

top_customers_min_payment = pay_rental.groupby(['first_name','last_name'], as_index=False) \
    .agg(
        avg_payment=('amount','mean'),
        total_payment=('amount','sum')
    ) \
    .sort_values('total_payment', ascending=True) \
    .head(10)

print(top_customers_min_payment)

    first_name last_name  avg_payment  total_payment
83    CAROLINE    BOWMAN     3.390000          50.85
351      LEONA    OBRIEN     3.632857          50.86
71       BRIAN     WYMAN     4.406667          52.88
296     JOHNNY    TURPIN     3.042632          57.81
33       ANNIE   RUSSELL     3.267778          58.82
319  KATHERINE    RIVERA     4.204286          58.86
550    TIFFANY    JORDAN     4.275714          59.86
28       ANITA   MORALES     4.190000          62.85
401     MATTIE   HOFFMAN     2.944545          64.78
334       KIRK   STCLAIR     3.411053          64.81


Soru 62: Mağazalardaki toplam kiralama süresini bulun.

In [165]:
rental_df['rental_date'] = pd.to_datetime(rental_df['rental_date'])
rental_df['return_date'] = pd.to_datetime(rental_df['return_date'])

store_rental = store_df.merge(staff_df, on='store_id') \
    .merge(rental_df, left_on='staff_id', right_on='staff_id')

store_rental['rental_days'] = (store_rental['return_date'] - store_rental['rental_date']).dt.days

total_rental_per_store = store_rental.groupby('store_id', as_index=False)['rental_days'].sum() \
    .rename(columns={'rental_days':'total_rental_days'})

print(total_rental_per_store)

   store_id  total_rental_days
0         1            35897.0
1         2            35889.0


Soru 63: En uzun süre kiralanan film hangi mağazada kiralanmış?

In [166]:
rental_df['rental_days'] = (rental_df['return_date'] - rental_df['rental_date']).dt.days
max_rental = rental_df.merge(staff_df[['staff_id','store_id']], on='staff_id') \
    .merge(inventory_df[['inventory_id','film_id']], on='inventory_id') \
    .merge(film_df[['film_id','title']], on='film_id') \
    .groupby(['store_id','title'], as_index=False)['rental_days'].max() \
    .sort_values('rental_days', ascending=False).head(1)
print(max_rental)


   store_id             title  rental_days
9         1  ALADDIN CALENDAR          9.0


Soru 64: En kısa süre kiralanan film hangi mağazada kiralanmış?

In [167]:
min_rental = rental_df.merge(staff_df[['staff_id','store_id']], on='staff_id') \
    .merge(inventory_df[['inventory_id','film_id']], on='inventory_id') \
    .merge(film_df[['film_id','title']], on='film_id') \
    .groupby(['store_id','title'], as_index=False)['rental_days'].min() \
    .sort_values('rental_days', ascending=True).head(1)
print(min_rental)

      store_id         title  rental_days
1898         2  WORDS HUNTER          0.0


Soru 65: Her filmin ortalama kiralama süresi nedir?

In [168]:
avg_rental = rental_df.merge(inventory_df[['inventory_id','film_id']], on='inventory_id') \
    .merge(film_df[['film_id','title']], on='film_id') \
    .groupby('title', as_index=False)['rental_days'].mean().rename(columns={'rental_days':'avg_rental_days'})
print(avg_rental)

                 title  avg_rental_days
0     ACADEMY DINOSAUR         4.545455
1       ACE GOLDFINGER         5.333333
2     ADAPTATION HOLES         2.833333
3     AFFAIR PREJUDICE         4.363636
4          AFRICAN EGG         6.727273
..                 ...              ...
953     YOUNG LANGUAGE         4.000000
954         YOUTH KICK         4.666667
955       ZHIVAGO CORE         5.250000
956  ZOOLANDER FICTION         5.176471
957          ZORRO ARK         4.096774

[958 rows x 2 columns]


Soru 66: Hangi filmler en çok kiralanan kategoride?

In [169]:
film_cat_full = film_df.merge(film_category_df, on='film_id').merge(category_df, on='category_id')
genre_rental_max = rental_df.merge(inventory_df[['inventory_id','film_id']], on='inventory_id') \
    .merge(film_cat_full[['film_id','title','name']], on='film_id') \
    .groupby(['name','title'], as_index=False).size().rename(columns={'size':'rental_count'}) \
    .sort_values('rental_count', ascending=False)
print(genre_rental_max.head())

        name                title  rental_count
909   Travel   BUCKET BROTHERHOOD            34
536  Foreign     ROCKETEER MOTHER            33
701    Music        SCALAWAG DUCK            32
757      New  RIDGEMONT SUBMARINE            32
568    Games       FORWARD TEMPLE            32


Soru 67: Hangi filmler en az kiralanan kategoride?

In [170]:
genre_rental_min = genre_rental_max.sort_values('rental_count', ascending=True).head()
print(genre_rental_min)

            name           title  rental_count
315  Documentary  HARDLY ROBBERS             4
525      Foreign     MIXED DOORS             4
656       Horror     TRAIN BUNCH             4
593        Games    PRIVATE DROP             5
597        Games     SEVEN SWARM             5


Soru 68: Hangi mağazalarda en çok film kiralanmış?

In [171]:
store_rental_count = rental_df.merge(staff_df[['staff_id','store_id']], on='staff_id') \
    .groupby('store_id', as_index=False).size().rename(columns={'size':'rental_count'}) \
    .sort_values('rental_count', ascending=False)
print(store_rental_count)

   store_id  rental_count
0         1          8040
1         2          8004


Soru 69: Hangi mağazalarda en az film kiralanmış?

In [172]:
store_rental_min = store_rental_count.sort_values('rental_count', ascending=True)
print(store_rental_min)

   store_id  rental_count
1         2          8004
0         1          8040


In [173]:
actor_film_count = film_actor_df.merge(actor_df, on='actor_id') \
    .groupby(['first_name','last_name'], as_index=False).size().rename(columns={'size':'film_count'}) \
    .sort_values('film_count', ascending=False)
print(actor_film_count.head(10))

    first_name    last_name  film_count
180      SUSAN        DAVIS          54
65        GINA    DEGENERES          42
190     WALTER         TORN          41
123       MARY       KEITEL          40
125    MATTHEW       CARREY          39
170     SANDRA       KILMER          37
173   SCARLETT        DAMON          36
78       HENRY        BERRY          35
72     GROUCHO        DUNST          35
8       ANGELA  WITHERSPOON          35


Soru 71: Hangi aktörler en az filmde rol almış?

In [174]:
actor_film_count = film_actor_df.merge(actor_df, on='actor_id') \
    .groupby(['first_name','last_name'], as_index=False) \
    .agg(film_count=('film_id','count')) \
    .sort_values('film_count', ascending=True)
print(actor_film_count)

    first_name  last_name  film_count
51       EMILY        DEE          14
101      JULIA    FAWCETT          15
99        JUDY       DEAN          15
103      JULIA  ZELLWEGER          16
0         ADAM      GRANT          18
..         ...        ...         ...
125    MATTHEW     CARREY          39
123       MARY     KEITEL          40
190     WALTER       TORN          41
65        GINA  DEGENERES          42
180      SUSAN      DAVIS          54

[199 rows x 3 columns]


Soru 72: Hangi aktörlerin oynadığı filmler en çok kiralanmış?

In [175]:
actor_rental_count = (
    film_actor_df.merge(actor_df, on='actor_id')[['actor_id','first_name','last_name','film_id']]
    .merge(film_df[['film_id']], on='film_id')
    .merge(inventory_df[['inventory_id','film_id']], on='film_id')
    .merge(rental_df[['rental_id','inventory_id']], on='inventory_id')
    .groupby(['first_name','last_name'], as_index=False)
    .agg(rental_count=('rental_id','count'))
    .sort_values('rental_count', ascending=False)
)

print(actor_rental_count)

    first_name    last_name  rental_count
180      SUSAN        DAVIS           825
65        GINA    DEGENERES           753
125    MATTHEW       CARREY           678
123       MARY       KEITEL           674
8       ANGELA  WITHERSPOON           654
..         ...          ...           ...
99        JUDY         DEAN           255
101      JULIA      FAWCETT           255
177      SISSY     SOBIESKI           235
103      JULIA    ZELLWEGER           221
51       EMILY          DEE           216

[199 rows x 3 columns]


Soru 73: Hangi aktörlerin oynadığı filmler en az kiralanmış?

In [176]:
actor_rental_min = (
    film_actor_df.merge(actor_df[['actor_id','first_name','last_name']], on='actor_id')
    .merge(film_df[['film_id']], on='film_id')
    .merge(inventory_df[['inventory_id','film_id']], on='film_id')
    .merge(rental_df[['rental_id','inventory_id']], on='inventory_id')
    .groupby(['first_name','last_name'], as_index=False)
    .agg(rental_count=('rental_id','count'))
    .sort_values('rental_count', ascending=True)
)

print(actor_rental_min)

    first_name    last_name  rental_count
51       EMILY          DEE           216
103      JULIA    ZELLWEGER           221
177      SISSY     SOBIESKI           235
99        JUDY         DEAN           255
101      JULIA      FAWCETT           255
..         ...          ...           ...
8       ANGELA  WITHERSPOON           654
123       MARY       KEITEL           674
125    MATTHEW       CARREY           678
65        GINA    DEGENERES           753
180      SUSAN        DAVIS           825

[199 rows x 3 columns]


Soru 74: Hangi kategorilerde en fazla aktör oynamış?

In [177]:
genre_actor_count = film_category_df.merge(category_df, on='category_id') \
    .merge(film_actor_df, on='film_id') \
    .groupby('name', as_index=False) \
    .agg(actor_count=('actor_id','nunique')) \
    .sort_values('actor_count', ascending=False)
print(genre_actor_count)

           name  actor_count
14       Sports          182
8       Foreign          175
12          New          169
5   Documentary          168
13       Sci-Fi          167
0        Action          166
1     Animation          166
15       Travel          166
7        Family          164
2      Children          163
3      Classics          162
6         Drama          162
10       Horror          156
9         Games          150
4        Comedy          147
11        Music          144


Soru 75: Hangi kategorilerde en az aktör oynamış?

In [178]:
category_actor_min = (
    film_category_df.merge(category_df[['category_id','name']], on='category_id')
    .merge(film_df[['film_id']], on='film_id')
    .merge(film_actor_df[['film_id','actor_id']], on='film_id')
    .groupby('name', as_index=False)
    .agg(actor_count=('actor_id','nunique'))
    .sort_values('actor_count', ascending=True)
)

print(category_actor_min)

           name  actor_count
11        Music          144
4        Comedy          147
9         Games          150
10       Horror          156
6         Drama          162
3      Classics          162
2      Children          163
7        Family          164
1     Animation          166
0        Action          166
15       Travel          166
13       Sci-Fi          167
5   Documentary          168
12          New          169
8       Foreign          175
14       Sports          182


Soru 76: Hangi kategorilerde oynayan filmler en çok kiralanmış?

In [179]:
genre_rental_count = (
    film_category_df.merge(category_df[['category_id','name']], on='category_id')
    .merge(film_df[['film_id']], on='film_id')
    .merge(inventory_df[['inventory_id','film_id']], on='film_id')
    .merge(rental_df[['rental_id','inventory_id']], on='inventory_id')
    .groupby('name', as_index=False)
    .agg(rental_count=('rental_id','count'))
    .sort_values('rental_count', ascending=False)
)

print(genre_rental_count)

           name  rental_count
14       Sports          1179
1     Animation          1166
0        Action          1112
13       Sci-Fi          1101
7        Family          1096
6         Drama          1060
5   Documentary          1050
8       Foreign          1033
9         Games           969
2      Children           945
4        Comedy           941
12          New           940
3      Classics           939
10       Horror           846
15       Travel           837
11        Music           830


Soru 77: Hangi kategorilerde oynayan filmler en az kiralanmış?

In [180]:
genre_rental_min = (
    film_category_df.merge(category_df[['category_id','name']], on='category_id')
    .merge(film_df[['film_id']], on='film_id')
    .merge(inventory_df[['inventory_id','film_id']], on='film_id')
    .merge(rental_df[['rental_id','inventory_id']], on='inventory_id')
    .groupby('name', as_index=False)
    .agg(rental_count=('rental_id','count'))
    .sort_values('rental_count', ascending=True)
)

print(genre_rental_min)

           name  rental_count
11        Music           830
15       Travel           837
10       Horror           846
3      Classics           939
12          New           940
4        Comedy           941
2      Children           945
9         Games           969
8       Foreign          1033
5   Documentary          1050
6         Drama          1060
7        Family          1096
13       Sci-Fi          1101
0        Action          1112
1     Animation          1166
14       Sports          1179


Soru 78: En fazla kazanç sağlayan kategoriler hangileri?

In [181]:
category_revenue_max = (
    film_category_df.merge(category_df[['category_id','name']], on='category_id')
    .merge(film_df[['film_id']], on='film_id')
    .merge(inventory_df[['inventory_id','film_id']], on='film_id')
    .merge(rental_df[['rental_id','inventory_id']], on='inventory_id')
    .merge(payment_df[['payment_id','rental_id','amount']], on='rental_id')
    .groupby('name', as_index=False)
    .agg(total_revenue=('amount','sum'))
    .sort_values('total_revenue', ascending=False)
)

print(category_revenue_max)

           name  total_revenue
14       Sports        5314.21
13       Sci-Fi        4756.98
1     Animation        4656.30
6         Drama        4587.39
4        Comedy        4383.58
0        Action        4375.85
12          New        4351.62
9         Games        4281.33
8       Foreign        4270.67
7        Family        4226.07
5   Documentary        4217.52
10       Horror        3722.54
2      Children        3655.55
3      Classics        3639.59
15       Travel        3549.64
11        Music        3417.72


Soru 79: En az kazanç sağlayan kategoriler hangileri?

In [182]:
category_revenue_min = (
    film_category_df.merge(category_df[['category_id','name']], on='category_id')
    .merge(film_df[['film_id']], on='film_id')
    .merge(inventory_df[['inventory_id','film_id']], on='film_id')
    .merge(rental_df[['rental_id','inventory_id']], on='inventory_id')
    .merge(payment_df[['payment_id','rental_id','amount']], on='rental_id')
    .groupby('name', as_index=False)
    .agg(total_revenue=('amount','sum'))
    .sort_values('total_revenue', ascending=True)
)

print(category_revenue_min)

           name  total_revenue
11        Music        3417.72
15       Travel        3549.64
3      Classics        3639.59
2      Children        3655.55
10       Horror        3722.54
5   Documentary        4217.52
7        Family        4226.07
8       Foreign        4270.67
9         Games        4281.33
12          New        4351.62
0        Action        4375.85
4        Comedy        4383.58
6         Drama        4587.39
1     Animation        4656.30
13       Sci-Fi        4756.98
14       Sports        5314.21


Soru 80: En çok film kiralayan müşterilerin ortalama ödeme miktarı nedir?

In [183]:
top_customers_avg_payment = (
    rental_df.merge(payment_df[['rental_id','amount']], on='rental_id')
    .merge(customer_df[['customer_id','first_name','last_name']], on='customer_id')
    .groupby(['first_name','last_name'], as_index=False)
    .agg(avg_payment=('amount','mean'), rental_count=('rental_id','count'))
    .sort_values('rental_count', ascending=False)
    .head(10)
)

print(top_customers_avg_payment)

    first_name last_name  avg_payment  rental_count
175    ELEANOR      HUNT     4.707391            46
318       KARL      SEAL     4.923333            45
379     MARCIA      DEAN     4.180476            42
105      CLARA      SHAW     4.656667            42
536      TAMMY   SANDERS     3.794878            41
590     WESLEY      BULL     4.440000            40
531        SUE    PETERS     3.865000            40
551        TIM      CARY     4.502821            39
389     MARION    SNYDER     4.990000            39
474     RHONDA   KENNEDY     4.990000            39


Soru 81: En az film kiralayan müşterilerin ortalama ödeme miktarı nedir?

In [184]:
customer_payment_min = (
    rental_df.merge(payment_df[['rental_id','amount']], on='rental_id')
    .merge(customer_df[['customer_id','first_name','last_name']], on='customer_id')
    .groupby(['first_name','last_name'], as_index=False)
    .agg(avg_payment=('amount','mean'), total_rentals=('rental_id','count'))
    .sort_values('total_rentals', ascending=True)
    .head(10)
)
print(customer_payment_min)

    first_name last_name  avg_payment  total_rentals
71       BRIAN     WYMAN     4.406667             12
550    TIFFANY    JORDAN     4.275714             14
351      LEONA    OBRIEN     3.632857             14
319  KATHERINE    RIVERA     4.204286             14
28       ANITA   MORALES     4.190000             15
83    CAROLINE    BOWMAN     3.390000             15
35     ANTONIO      MEEK     4.927500             16
356     LESTER     KRAUS     4.115000             16
277     JEROME    KENYON     4.615000             16
290      JOANN   GARDNER     4.177500             16


Soru 82: Hangi mağazalarda hangi kategoriler en çok kiralanmış?

In [185]:
store_genre_max = (
    staff_df.merge(store_df[['store_id']], on='store_id')
    .merge(rental_df[['rental_id','staff_id','inventory_id']], on='staff_id')
    .merge(inventory_df[['inventory_id','film_id']], on='inventory_id')
    .merge(film_df[['film_id']], on='film_id')
    .merge(film_category_df[['film_id','category_id']], on='film_id')
    .merge(category_df[['category_id','name']], on='category_id')
    .groupby(['store_id','name'], as_index=False)
    .agg(rental_count=('rental_id','count'))
    .sort_values('rental_count', ascending=False)
)
print(store_genre_max)

    store_id         name  rental_count
30         2       Sports           614
17         2    Animation           584
1          1    Animation           582
7          1       Family           565
14         1       Sports           565
0          1       Action           562
13         1       Sci-Fi           553
16         2       Action           550
29         2       Sci-Fi           548
24         2      Foreign           543
22         2        Drama           532
23         2       Family           531
6          1        Drama           528
21         2  Documentary           525
5          1  Documentary           525
9          1        Games           494
8          1      Foreign           490
12         1          New           489
2          1     Children           483
25         2        Games           475
3          1     Classics           475
4          1       Comedy           475
20         2       Comedy           466
19         2     Classics           464


Soru 83: Hangi mağazalarda hangi kategoriler en az kiralanmış?

In [186]:
store_genre_min = (
    staff_df.merge(store_df[['store_id']], on='store_id')
    .merge(rental_df[['rental_id','staff_id','inventory_id']], on='staff_id')
    .merge(inventory_df[['inventory_id','film_id']], on='inventory_id')
    .merge(film_df[['film_id']], on='film_id')
    .merge(film_category_df[['film_id','category_id']], on='film_id')
    .merge(category_df[['category_id','name']], on='category_id')
    .groupby(['store_id','name'], as_index=False)
    .agg(rental_count=('rental_id','count'))
    .sort_values('rental_count', ascending=True)   # ASC sıralama
)
print(store_genre_min)

    store_id         name  rental_count
27         2        Music           398
15         1       Travel           399
26         2       Horror           423
10         1       Horror           423
11         1        Music           432
31         2       Travel           438
28         2          New           451
18         2     Children           462
19         2     Classics           464
20         2       Comedy           466
25         2        Games           475
3          1     Classics           475
4          1       Comedy           475
2          1     Children           483
12         1          New           489
8          1      Foreign           490
9          1        Games           494
21         2  Documentary           525
5          1  Documentary           525
6          1        Drama           528
23         2       Family           531
22         2        Drama           532
24         2      Foreign           543
29         2       Sci-Fi           548


Soru 84: En fazla sayıda film kiralayan mağaza hangisidir ve toplam kiralama sayısı nedir?

In [187]:
store_rental_total = (
    staff_df.merge(rental_df[['rental_id','staff_id']], on='staff_id')
    .groupby('store_id', as_index=False)
    .agg(total_rentals=('rental_id','count'))
    .sort_values('total_rentals', ascending=False)
)
print(store_rental_total.head(1))

   store_id  total_rentals
0         1           8040


Soru 85: En az sayıda film kiralayan mağaza hangisidir ve toplam kiralama sayısı nedir?

In [188]:
store_rental_total = (
    staff_df.merge(rental_df[['rental_id','staff_id']], on='staff_id')
    .groupby('store_id', as_index=False)
    .agg(total_rentals=('rental_id','count'))
    .sort_values('total_rentals', ascending=True)   # ASC sıralama
)
print(store_rental_total.head(1))

   store_id  total_rentals
1         2           8004


Soru 86: En fazla sayıda kiralama yapan müşteri hangisidir ve toplam kiralama sayısı nedir?

In [189]:
customer_rental_total = (
    rental_df.merge(customer_df[['customer_id','first_name','last_name']], on='customer_id')
    .groupby(['first_name','last_name'], as_index=False)
    .agg(total_rentals=('rental_id','count'))
    .sort_values('total_rentals', ascending=False)
)
print(customer_rental_total.head(1))

    first_name last_name  total_rentals
175    ELEANOR      HUNT             46


Soru 87: En az sayıda kiralama yapan müşteri hangisidir ve toplam kiralama sayısı nedir? 

In [190]:
customer_rental_total = (
    rental_df.merge(customer_df[['customer_id','first_name','last_name']], on='customer_id')
    .groupby(['first_name','last_name'], as_index=False)
    .agg(total_rentals=('rental_id','count'))
    .sort_values('total_rentals', ascending=True)  # ASC sıralama
)
print(customer_rental_total.head(1))

   first_name last_name  total_rentals
71      BRIAN     WYMAN             12


Soru 88: En fazla sayıda kiralanan film hangisidir ve toplam kiralama sayısı nedir?

In [191]:
film_rental_total = (
    rental_df.merge(inventory_df[['inventory_id','film_id']], on='inventory_id')
    .merge(film_df[['film_id','title']], on='film_id')
    .groupby('title', as_index=False)
    .agg(total_rentals=('rental_id','count'))
    .sort_values('total_rentals', ascending=False)
)
print(film_rental_total.head(1))

                 title  total_rentals
96  BUCKET BROTHERHOOD             34


Soru 89: En az sayıda kiralanan film hangisidir ve toplam kiralama sayısı nedir?

In [192]:
film_rental_total = (
    rental_df.merge(inventory_df[['inventory_id','film_id']], on='inventory_id')
    .merge(film_df[['film_id','title']], on='film_id')
    .groupby('title', as_index=False)
    .agg(total_rentals=('rental_id','count'))
    .sort_values('total_rentals', ascending=True)  # ASC sıralama
)
print(film_rental_total.head(1))

           title  total_rentals
866  TRAIN BUNCH              4


Soru 90: En fazla kazanç sağlayan müşteri hangisidir ve toplam kazancı nedir?

In [193]:
customer_revenue = (
    rental_df.merge(payment_df[['rental_id','amount']], on='rental_id')
    .merge(customer_df[['customer_id','first_name','last_name']], on='customer_id')
    .groupby(['first_name','last_name'], as_index=False)
    .agg(total_revenue=('amount','sum'))
    .sort_values('total_revenue', ascending=False)
)
print(customer_revenue.head(1))

    first_name last_name  total_revenue
318       KARL      SEAL         221.55


Soru 91: En az kazanç sağlayan müşteri hangisidir ve toplam kazancı nedir?

In [194]:
customer_revenue_min = (
    rental_df.merge(customer_df[['customer_id','first_name','last_name']], on='customer_id')
    .merge(payment_df[['rental_id','amount']], on='rental_id')
    .groupby(['first_name','last_name'], as_index=False)
    .agg(total_revenue=('amount','sum'))
    .sort_values('total_revenue', ascending=True)
)
print(customer_revenue_min.head(1))

   first_name last_name  total_revenue
83   CAROLINE    BOWMAN          50.85


Soru 92: Hangi film, hangi kategoride en çok kazanç sağlamış?

In [195]:
film_genre_revenue_max = (
    film_category_df.merge(film_df[['film_id','title']], on='film_id')
    .merge(category_df[['category_id','name']], on='category_id')
    .merge(inventory_df[['inventory_id','film_id']], on='film_id')
    .merge(rental_df[['rental_id','inventory_id']], on='inventory_id')
    .merge(payment_df[['rental_id','amount']], on='rental_id')
    .groupby(['title','name'], as_index=False)
    .agg(total_revenue=('amount','sum'))
    .sort_values('total_revenue', ascending=False)
)
print(film_genre_revenue_max.head(1))

                title   name  total_revenue
841  TELEGRAPH VOYAGE  Music         231.73


Soru 93: Hangi film, hangi kategoride en az kazanç sağlamış?

In [196]:
film_genre_revenue_min = (
    film_category_df.merge(film_df[['film_id','title']], on='film_id')
    .merge(category_df[['category_id','name']], on='category_id')
    .merge(inventory_df[['inventory_id','film_id']], on='film_id')
    .merge(rental_df[['rental_id','inventory_id']], on='inventory_id')
    .merge(payment_df[['rental_id','amount']], on='rental_id')
    .groupby(['title','name'], as_index=False)
    .agg(total_revenue=('amount','sum'))
    .sort_values('total_revenue', ascending=True)
)
print(film_genre_revenue_min.head(1))

           title    name  total_revenue
847  TEXAS WATCH  Horror           5.94


Soru 94: En fazla kazanç sağlayan mağaza hangisidir ve toplam kazancı nedir?

In [197]:
store_revenue_max = (
    staff_df.merge(store_df[['store_id']], on='store_id')
    .merge(rental_df[['rental_id','staff_id']], on='staff_id')
    .merge(payment_df[['rental_id','amount']], on='rental_id')
    .groupby('store_id', as_index=False)
    .agg(total_revenue=('amount','sum'))
    .sort_values('total_revenue', ascending=False)
)
print(store_revenue_max.head(1))

   store_id  total_revenue
1         2       33881.94


Soru 95: En az kazanç sağlayan mağaza hangisidir ve toplam kazancı nedir?

In [198]:
store_revenue_min = (
    staff_df.merge(store_df[['store_id']], on='store_id')
    .merge(rental_df[['rental_id','staff_id']], on='staff_id')
    .merge(payment_df[['rental_id','amount']], on='rental_id')
    .groupby('store_id', as_index=False)
    .agg(total_revenue=('amount','sum'))
    .sort_values('total_revenue', ascending=True)
)
print(store_revenue_min.head(1))

   store_id  total_revenue
0         1       33524.62


Soru 96: En fazla sayıda farklı film kiralayan müşteri hangisidir?

In [199]:
unique_rentals_max = (
    rental_df.merge(customer_df[['customer_id','first_name','last_name']], on='customer_id')
    .groupby(['first_name','last_name'], as_index=False)
    .agg(unique_rentals=('inventory_id','nunique'))
    .sort_values('unique_rentals', ascending=False)
)
print(unique_rentals_max.head(1))

    first_name last_name  unique_rentals
175    ELEANOR      HUNT              46


Soru 97: En az sayıda farklı film kiralayan müşteri hangisidir?

In [200]:
unique_rentals_min = (
    rental_df.merge(customer_df[['customer_id','first_name','last_name']], on='customer_id')
    .groupby(['first_name','last_name'], as_index=False)
    .agg(unique_rentals=('inventory_id','nunique'))
    .sort_values('unique_rentals', ascending=True)
)
print(unique_rentals_min.head(1))

   first_name last_name  unique_rentals
71      BRIAN     WYMAN              12


Soru 98: En fazla sayıda farklı müşteri tarafından kiralanan film hangisidir?

In [201]:
unique_customers_max = (
    inventory_df.merge(film_df[['film_id','title']], on='film_id')
    .merge(rental_df[['inventory_id','customer_id']], on='inventory_id')
    .groupby('title', as_index=False)
    .agg(unique_customers=('customer_id','nunique'))
    .sort_values('unique_customers', ascending=False)
)
print(unique_customers_max.head(1))

                 title  unique_customers
96  BUCKET BROTHERHOOD                33


Soru 99: En az sayıda farklı müşteri tarafından kiralanan film hangisidir?

In [202]:
unique_customers_min = (
    inventory_df.merge(film_df[['film_id','title']], on='film_id')
    .merge(rental_df[['inventory_id','customer_id']], on='inventory_id')
    .groupby('title', as_index=False)
    .agg(unique_customers=('customer_id','nunique'))
    .sort_values('unique_customers', ascending=True)
)
print(unique_customers_min.head(1))

            title  unique_customers
417  HUNTER ALTER                 4


Soru 100: Hangi mağazada, hangi kategoride en fazla kazanç sağlanmış?

In [203]:
store_genre_revenue_max = (
    staff_df.merge(store_df[['store_id']], on='store_id')
    .merge(rental_df[['rental_id','staff_id','inventory_id']], on='staff_id')
    .merge(inventory_df[['inventory_id','film_id']], on='inventory_id')
    .merge(film_df[['film_id']], on='film_id')
    .merge(film_category_df[['film_id','category_id']], on='film_id')
    .merge(category_df[['category_id','name']], on='category_id')
    .merge(payment_df[['rental_id','amount']], on='rental_id')
    .groupby(['store_id','name'], as_index=False)
    .agg(total_revenue=('amount','sum'))
    .sort_values('total_revenue', ascending=False)
)
print(store_genre_revenue_max.head(1))

    store_id    name  total_revenue
30         2  Sports        2761.85
